# 13 — Structured Output

Search the web for any subject and extract a validated **Pydantic profile** using `with_structured_output()`.

**What you'll learn**
- `with_structured_output(MyModel)` forces the LLM to return data that parses into your Pydantic model — no string parsing, no JSON hacking
- The search-then-extract two-node pattern: raw retrieval separated from structured generation
- How `Field(description=...)` guides what the LLM fills in each field
- DuckDuckGo as a zero-key-required web search tool

**Companion examples:** 29-llm-judge-harness (structured rubric scoring), 25-adaptive-rag (structured routing decision), 27-self-rag (BoolDecision gates)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-community duckduckgo-search python-dotenv langgraph pydantic

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Define the output schema ───────────────────────────────────────────────
# Field descriptions are read by the LLM — they guide what gets extracted.
# The LLM MUST return data matching this shape; Pydantic validates it.
from pydantic import BaseModel, Field


class EntityProfile(BaseModel):
    name: str = Field(description="Full name of the entity")
    summary: str = Field(description="2-3 sentence overview")
    key_facts: list[str] = Field(description="3-5 notable facts")
    confidence: str = Field(description="high | medium | low based on source quality")


print("Schema defined — LLM must return: name, summary, key_facts, confidence")

In [ ]:
# ── 4. Build the search → extract graph ───────────────────────────────────────
# Two nodes, clean separation of concerns:
#   search : raw DuckDuckGo text → state["raw_results"]
#   extract: structured LLM      → state["profile"] (validated Pydantic object)
import json
from typing import TypedDict

from duckduckgo_search import DDGS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


class ProfileState(TypedDict):
    subject: str
    raw_results: str
    profile: dict


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(EntityProfile)  # ← the key line


def search_node(state: ProfileState) -> dict:
    with DDGS() as ddgs:
        results = list(ddgs.text(state["subject"], max_results=5))
    text = "\n".join(f"{r['title']}: {r['body']}" for r in results) if results else "No results."
    return {"raw_results": text}


def extract_node(state: ProfileState) -> dict:
    profile = structured_llm.invoke([
        SystemMessage(f"Extract a structured profile for: {state['subject']}"),
        HumanMessage(f"Use these search results:\n{state['raw_results']}"),
    ])
    return {"profile": profile.model_dump()}


graph = StateGraph(ProfileState)
graph.add_node("search", search_node)
graph.add_node("extract", extract_node)
graph.add_edge(START, "search")
graph.add_edge("search", "extract")
graph.add_edge("extract", END)
app = graph.compile()

print("Graph: START → search → extract → END")

In [ ]:
# ── 5. Profile any subject ─────────────────────────────────────────────────────
SUBJECTS = ["LangChain", "Andrej Karpathy"]

for subject in SUBJECTS:
    print(f"\nProfiling: {subject}")
    print("─" * 50)
    result = app.invoke({"subject": subject, "raw_results": "", "profile": {}})
    print(json.dumps(result["profile"], indent=2))

## Exercises

**Exercise 1 — Add a field:** Add a `founded_year: int | None` field to `EntityProfile`. Run it on `"Anthropic"`. Does the LLM fill it correctly?

**Exercise 2 — Profile an abstract concept:** Try `"Retrieval Augmented Generation"` or `"LangGraph"`. How does the LLM handle concepts vs. named entities?

**Exercise 3 — Enforce a Literal type:** Change `confidence: str` to `confidence: Literal["high", "medium", "low"]` from `typing`. Does the LLM still comply?

## What's next

- **29-llm-judge-harness** — structured output for rubric scoring (0-3 on Relevance, Faithfulness, Completeness)
- **25-adaptive-rag** — `RouteDecision(strategy: Literal[...])` chooses retrieval path per query
- **27-self-rag** — `BoolDecision(answer: bool)` powers three LLM decision gates